# ML Model Development for Pairs Trading

This notebook demonstrates how to build and train machine learning models for pairs trading signals.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import yaml

# Import project modules
import sys
sys.path.append('../..')
from src.feature_engineering import create_spread_features, create_lag_features
from src.trading_strategy import PairsTradingStrategy

%matplotlib inline

## 1. Load Configuration

In [ ]:
# Load configuration
with open('../../configs/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully")
print(f"Model type: {config['model']['type']}")

## 2. Prepare Data and Features

Create features from spread data for ML model training.

In [ ]:
# This is a placeholder - in practice, you would load actual data
# For demonstration purposes only

# Example: Generate synthetic spread data
np.random.seed(42)
dates = pd.date_range('2020-01-01', periods=1000, freq='D')
spread = pd.Series(np.cumsum(np.random.randn(1000) * 0.5), index=dates)

print(f"Spread data shape: {spread.shape}")
spread.head()

In [ ]:
# Create features
spread_features = create_spread_features(spread, windows=config['features']['spread_windows'])
lag_features = create_lag_features(spread, lags=config['features']['lag_periods'])

# Combine all features
features = pd.concat([spread_features, lag_features], axis=1)
features = features.dropna()

print(f"Feature matrix shape: {features.shape}")
print(f"\nFeatures: {list(features.columns)}")

## 3. Generate Labels

Create labels based on future returns (profitable signals).

In [ ]:
# Calculate future returns (5 days ahead)
future_returns = spread.shift(-5) - spread

# Create labels: 1 if positive return, 0 otherwise
labels = (future_returns > 0).astype(int)

# Align with features
labels = labels.loc[features.index]

print(f"Label distribution:\n{labels.value_counts()}")
print(f"Positive label percentage: {labels.mean():.2%}")

## 4. Train-Test Split

In [ ]:
# Split data
split_ratio = config['model']['train_test_split']
split_idx = int(len(features) * split_ratio)

X_train = features.iloc[:split_idx]
X_test = features.iloc[split_idx:]
y_train = labels.iloc[:split_idx]
y_test = labels.iloc[split_idx:]

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTraining label distribution:\n{y_train.value_counts()}")
print(f"Test label distribution:\n{y_test.value_counts()}")

## 5. Train Model

In [ ]:
# Initialize model
model = RandomForestClassifier(
    n_estimators=config['model']['hyperparameters']['n_estimators'],
    max_depth=config['model']['hyperparameters']['max_depth'],
    random_state=config['model']['hyperparameters']['random_state']
)

# Train model
print("Training model...")
model.fit(X_train, y_train)
print("Model trained successfully!")

## 6. Evaluate Model

In [ ]:
# Make predictions
y_pred_train = model.predict(X_train)
y_pred_test = model.predict(X_test)

# Calculate accuracy
train_accuracy = (y_pred_train == y_train).mean()
test_accuracy = (y_pred_test == y_test).mean()

print(f"Training Accuracy: {train_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Classification report
print("\nClassification Report (Test Set):")
print(classification_report(y_test, y_pred_test))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Trade', 'Trade'],
            yticklabels=['No Trade', 'Trade'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Feature Importance

In [ ]:
# Get feature importance
feature_importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

# Plot top 15 features
plt.figure(figsize=(10, 8))
top_features = feature_importance.head(15)
plt.barh(range(len(top_features)), top_features['importance'])
plt.yticks(range(len(top_features)), top_features['feature'])
plt.xlabel('Importance', fontsize=12)
plt.title('Top 15 Feature Importances', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

## 8. Save Model

In [ ]:
# Save model
if config['output']['save_models']:
    import joblib
    model_path = '../../models/random_forest_model.pkl'
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path}")

## Next Steps

1. Hyperparameter tuning using cross-validation
2. Try different ML algorithms (XGBoost, LightGBM, Neural Networks)
3. Implement online learning for adaptive models
4. Integrate model predictions with trading strategy
5. Backtest the ML-enhanced strategy